In [14]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

tickers = ["AAPL","MSFT","GOOGL","AMZN","META","JPM","BAC","C","WFC","XOM","CVX",]
data = yf.download(tickers, start="2022-01-01", end="2024-01-01")
price_df = data["Close"].dropna(axis=1, how="all").dropna()



def find_cointegrated_pairs(price_df, p_value_threshold=0.05):
    pairs = []

    for stock1 in price_df.columns:
        for stock2 in price_df.columns:
            if stock1 < stock2:
                s1 = price_df[stock1]
                s2 = price_df[stock2]

                result = coint(s1, s2)
                pvalue = result[1]

                if pvalue < p_value_threshold:
                    pairs.append((stock1, stock2, pvalue))
                    print("Pair:", stock1, "&", stock2, "| p-value:", round(pvalue, 4))

        

    return sorted(pairs, key=lambda x: x[2])



cointegrated_pairs = find_cointegrated_pairs(price_df)

results = []

for stock1, stock2, p_val in cointegrated_pairs:
    x = price_df[[stock2]]
    y = price_df[stock1]

    model = LinearRegression().fit(x, y)
    beta = model.coef_[0]

    spread = y - (beta * price_df[stock2])
    z_score = (spread - spread.mean()) / spread.std()
    latest_z = z_score.iloc[-1] 

    
    if latest_z > 2.0:
        signal = "SHORT " + stock1 + " / LONG " + stock2
    elif latest_z < -2.0:
        signal = "LONG " + stock1 + " / SHORT " + stock2
    else:
        signal = "NEUTRAL"

    results.append({"Stock 1": stock1,"Stock 2": stock2,"p-value": round(p_val, 4),"Beta": round(beta, 4),"Current Z-Score": round(latest_z, 2),"Signal": signal,})


results_df = pd.DataFrame(results)

print(results_df)

 
    

        




[*********************100%***********************]  11 of 11 completed


Pair: AAPL & MSFT | p-value: 0.025
Pair: CVX & GOOGL | p-value: 0.0279
Pair: CVX & JPM | p-value: 0.039
Pair: CVX & META | p-value: 0.0499
Pair: CVX & MSFT | p-value: 0.0359
Pair: CVX & WFC | p-value: 0.0215
Pair: META & MSFT | p-value: 0.0188
  Stock 1 Stock 2  p-value    Beta  Current Z-Score   Signal
0    META    MSFT   0.0188  1.6485            -0.35  NEUTRAL
1     CVX     WFC   0.0215 -1.0491             0.17  NEUTRAL
2    AAPL    MSFT   0.0250  0.4075            -0.91  NEUTRAL
3     CVX   GOOGL   0.0279 -0.3460             0.35  NEUTRAL
4     CVX    MSFT   0.0359 -0.0799             0.24  NEUTRAL
5     CVX     JPM   0.0390 -0.0380            -0.23  NEUTRAL
6     CVX    META   0.0499 -0.0584             0.32  NEUTRAL
